# Notification Logic

This notebook builds live airport notification logic: event ingestion, severity scoring, category routing, deduplication, read/dismiss state, escalation rules, alert feed filtering, analytics, and backend-ready payloads for the Command Center notification module.

## 1. Setup

In [ ]:
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

now = datetime(2026, 4, 27, 9, 0)

## 2. Notification Configuration

In [ ]:
SEVERITY_CONFIG = {
    'critical': {'weight': 100, 'sla_min': 3, 'requires_ack': True, 'route': ['Airport Duty Manager', 'ATC Supervisor', 'Emergency Response']},
    'high': {'weight': 75, 'sla_min': 8, 'requires_ack': True, 'route': ['Operations Lead', 'Terminal Control']},
    'medium': {'weight': 45, 'sla_min': 20, 'requires_ack': False, 'route': ['Shift Supervisor']},
    'low': {'weight': 20, 'sla_min': 60, 'requires_ack': False, 'route': ['Module Owner']},
}

CATEGORY_CONFIG = {
    'weather': {'label': 'Weather', 'owner': 'Met Desk'},
    'runway': {'label': 'Runway', 'owner': 'Airside Ops'},
    'equipment': {'label': 'Equipment', 'owner': 'GSE Control'},
    'security': {'label': 'Security', 'owner': 'Airport Security'},
    'fuel': {'label': 'Fuel', 'owner': 'Fuel Farm'},
    'flight': {'label': 'Flight Ops', 'owner': 'Flight Operations'},
    'system': {'label': 'System', 'owner': 'IT Operations'},
}

ALERT_TEMPLATES = [
    {'sev': 'critical', 'cat': 'weather', 'title': 'Severe Crosswind Exceeded', 'body': 'Crosswind component 22kt on RWY 28L exceeds aircraft limits. Suspend ops immediately.'},
    {'sev': 'critical', 'cat': 'runway', 'title': 'FOD Detected - RWY 10R', 'body': 'Foreign object debris confirmed near touchdown zone. RWY 10R closed pending inspection.'},
    {'sev': 'critical', 'cat': 'security', 'title': 'Unauthorized Vehicle on Apron', 'body': 'Vehicle ID UNK-447 entered restricted Zone C without clearance. Security dispatched.'},
    {'sev': 'critical', 'cat': 'fuel', 'title': 'Fuel Bowser Leak Detected', 'body': 'Fuel spill alert at Stand 14. Emergency response team activated. Area cordoned.'},
    {'sev': 'high', 'cat': 'weather', 'title': 'Low Visibility Warning', 'body': 'Visibility dropped to 3,200m at runway threshold. CAT I minima in effect.'},
    {'sev': 'high', 'cat': 'equipment', 'title': 'Ground Power Unit Failure', 'body': 'GPU Unit 7 at Stand 22 offline. Aircraft EI-DVM has no ground power. Spare dispatched.'},
    {'sev': 'high', 'cat': 'flight', 'title': 'Pushback Delay - Stand 31', 'body': 'TUG-09 tow bar failure. AI-234 departure delayed 18 min. Alternate tug assigned.'},
    {'sev': 'high', 'cat': 'runway', 'title': 'Brake Fluid Detected on Taxiway B', 'body': 'Maintenance crew en route. Taxiway B partially restricted pending cleanup.'},
    {'sev': 'high', 'cat': 'system', 'title': 'FIDS Outage - Terminal 2', 'body': 'Flight information displays in T2 departure hall offline. IT team notified. ETA fix: 15 min.'},
    {'sev': 'medium', 'cat': 'weather', 'title': 'Wind Shear Advisory', 'body': 'Moderate wind shear reported at 1,200ft on approach. Crews alerted.'},
    {'sev': 'medium', 'cat': 'equipment', 'title': 'Conveyor Belt Fault - Belt 5', 'body': 'Baggage belt 5 intermittent fault. Manual handling activated as contingency.'},
    {'sev': 'medium', 'cat': 'fuel', 'title': 'Fuel Truck Queue Exceeding 12 min', 'body': 'High turnaround demand. 3 aircraft waiting for refuelling at North Apron.'},
    {'sev': 'medium', 'cat': 'flight', 'title': 'Gate Change - Flight AI-109', 'body': 'Gate changed from A12 to B07. Ramp crew and catering redirected.'},
    {'sev': 'medium', 'cat': 'security', 'title': 'Passenger Screened - Secondary', 'body': 'Passenger requiring secondary screening at Gate C4. Departure may be impacted.'},
    {'sev': 'low', 'cat': 'system', 'title': 'Routine Backup Completed', 'body': 'Nightly database backup completed successfully at 02:14 UTC.'},
    {'sev': 'low', 'cat': 'equipment', 'title': 'Scheduled Maintenance - TUG-03', 'body': 'TUG-03 due for 500hr service at 06:00. Replacement TUG-11 pre-positioned.'},
    {'sev': 'low', 'cat': 'weather', 'title': 'Cloud Ceiling Update', 'body': 'Ceiling raised to 3,400ft. IFR restrictions lifted on RWY 10L.'},
    {'sev': 'low', 'cat': 'flight', 'title': 'Early Arrival - Flight EK-512', 'body': 'EK-512 estimated 9 min early. Stand 17 prep teams alerted.'},
]

## 3. Generate Incoming Event Stream

In [ ]:
def make_event(event_id, timestamp):
    template = random.choices(
        ALERT_TEMPLATES,
        weights=[7 if t['sev'] == 'critical' else 13 if t['sev'] == 'high' else 22 if t['sev'] == 'medium' else 18 for t in ALERT_TEMPLATES],
        k=1,
    )[0]
    source_map = {
        'weather': 'weather_engine',
        'runway': 'airside_sensor',
        'equipment': 'gse_telemetry',
        'security': 'access_control',
        'fuel': 'fuel_ops',
        'flight': 'flight_ops',
        'system': 'it_monitoring',
    }
    confidence = round(np.clip(np.random.normal(0.88, 0.09), 0.55, 0.99), 2)
    impact_score = int(np.clip(np.random.normal(SEVERITY_CONFIG[template['sev']]['weight'], 12), 5, 100))
    duplicate_noise = random.random() < 0.12
    title = template['title'] if not duplicate_noise else random.choice([template['title'], template['title'] + ' Update'])
    return {
        'event_id': f"EVT-{event_id:04d}",
        'timestamp': timestamp,
        'source': source_map[template['cat']],
        'cat': template['cat'],
        'detected_sev': template['sev'],
        'title': title,
        'body': template['body'],
        'location': random.choice(['Terminal 1', 'Terminal 2', 'North Apron', 'South Apron', 'RWY 10R', 'RWY 28L', 'Stand 14', 'Gate C4']),
        'confidence': confidence,
        'impact_score': impact_score,
        'affected_flights': int(np.clip(np.random.poisson(2 if template['sev'] in ['critical', 'high'] else 1), 0, 12)),
    }

events = []
for i in range(80):
    ts = now - timedelta(minutes=int(np.random.uniform(0, 180)))
    events.append(make_event(i + 1, ts))

events_df = pd.DataFrame(events).sort_values('timestamp', ascending=False).reset_index(drop=True)
events_df.head(10)

## 4. Alert Scoring and Severity Normalization

In [ ]:
def normalize_severity(row):
    score = row['impact_score']
    score += row['affected_flights'] * 2.5
    score += 8 if row['confidence'] >= 0.92 else 0
    score -= 10 if row['confidence'] < 0.7 else 0
    if row['cat'] in ['security', 'fuel', 'runway'] and score >= 65:
        score += 8
    score = round(min(100, max(0, score)), 1)
    if score >= 82:
        sev = 'critical'
    elif score >= 62:
        sev = 'high'
    elif score >= 35:
        sev = 'medium'
    else:
        sev = 'low'
    return pd.Series({'priority_score': score, 'sev': sev})

scored_df = pd.concat([events_df, events_df.apply(normalize_severity, axis=1)], axis=1)
scored_df[['event_id', 'timestamp', 'cat', 'detected_sev', 'sev', 'priority_score', 'title', 'affected_flights']].head(10)

## 5. Deduplication and Alert Feed State

In [ ]:
def dedupe_key(row):
    normalized_title = row['title'].replace(' Update', '').lower().strip()
    return f"{row['cat']}::{row['location']}::{normalized_title}"


def collapse_duplicate_window(df, window_minutes=20):
    collapsed_rows = []
    ordered = df.sort_values(['dedupe_key', 'timestamp'], ascending=[True, False])
    for _, group in ordered.groupby('dedupe_key', sort=False):
        cluster = []
        anchor_time = None
        for record in group.to_dict(orient='records'):
            timestamp = record['timestamp']
            if anchor_time is None or (anchor_time - timestamp).total_seconds() / 60 > window_minutes:
                if cluster:
                    latest = cluster[0].copy()
                    latest['duplicate_count'] = len(cluster)
                    latest['duplicate_event_ids'] = [item['event_id'] for item in cluster]
                    collapsed_rows.append(latest)
                cluster = [record]
                anchor_time = timestamp
            else:
                cluster.append(record)
        if cluster:
            latest = cluster[0].copy()
            latest['duplicate_count'] = len(cluster)
            latest['duplicate_event_ids'] = [item['event_id'] for item in cluster]
            collapsed_rows.append(latest)
    return pd.DataFrame(collapsed_rows)


feed_df = scored_df.copy()
feed_df['dedupe_key'] = feed_df.apply(dedupe_key, axis=1)
feed_df = collapse_duplicate_window(feed_df, window_minutes=20)
feed_df = feed_df.sort_values(['priority_score', 'timestamp'], ascending=[False, False]).reset_index(drop=True)
feed_df['alert_id'] = [f"ALT-{i:04d}" for i in range(1, len(feed_df) + 1)]
feed_df['read'] = feed_df['timestamp'] < now - timedelta(minutes=45)
feed_df['acknowledged'] = False
feed_df['dismissed'] = False
feed_df['fresh'] = feed_df['timestamp'] >= now - timedelta(minutes=5)
feed_df['age_min'] = ((now - feed_df['timestamp']).dt.total_seconds() / 60).round(1)
feed_df[['alert_id', 'sev', 'cat', 'title', 'location', 'duplicate_count', 'read', 'fresh', 'age_min']].head(12)


## 6. Routing and Escalation

In [ ]:
def route_alert(row):
    sev_cfg = SEVERITY_CONFIG[row['sev']]
    cat_cfg = CATEGORY_CONFIG[row['cat']]
    overdue = row['age_min'] > sev_cfg['sla_min'] and ((sev_cfg['requires_ack'] and not row['acknowledged']) or not row['read'])
    escalation_status = 'escalate' if overdue else 'ack_pending' if sev_cfg['requires_ack'] and not row['acknowledged'] else 'within_sla'
    return pd.Series({
        'owner': cat_cfg['owner'],
        'route_to': ', '.join(sev_cfg['route']),
        'sla_min': sev_cfg['sla_min'],
        'requires_ack': sev_cfg['requires_ack'],
        'sla_remaining_min': round(sev_cfg['sla_min'] - row['age_min'], 1),
        'escalation_status': escalation_status,
    })


feed_df = pd.concat([feed_df, feed_df.apply(route_alert, axis=1)], axis=1)
escalations_df = feed_df[feed_df['escalation_status'] == 'escalate'].copy()
feed_df[['alert_id', 'sev', 'cat', 'owner', 'route_to', 'sla_min', 'age_min', 'escalation_status']].head(12)


## 7. Feed Filters and Search Logic

In [ ]:
def filter_alerts(df, severities=None, category='all', search=''):
    filtered = df[~df['dismissed']].copy()
    if severities:
        filtered = filtered[filtered['sev'].isin(severities)]
    if category != 'all':
        filtered = filtered[filtered['cat'] == category]
    if search:
        term = search.lower()
        filtered = filtered[
            filtered['title'].str.lower().str.contains(term, regex=False) |
            filtered['body'].str.lower().str.contains(term, regex=False)
        ]
    return filtered.sort_values(['priority_score', 'timestamp'], ascending=[False, False])


visible_alerts_df = filter_alerts(feed_df, severities=['critical', 'high', 'medium', 'low'], category='all')
critical_weather_df = filter_alerts(feed_df, severities=['critical', 'high'], category='weather')
visible_alerts_df[['alert_id', 'sev', 'cat', 'title', 'read', 'priority_score']].head(10)


## 8. Read, Dismiss, and Live Insert Actions

In [ ]:
def refresh_alert_routing(df):
    refreshed = df.drop(columns=['owner', 'route_to', 'sla_min', 'requires_ack', 'sla_remaining_min', 'escalation_status'], errors='ignore').copy()
    return pd.concat([refreshed, refreshed.apply(route_alert, axis=1)], axis=1)


def mark_read(df, alert_ids):
    updated = df.copy()
    updated.loc[updated['alert_id'].isin(alert_ids), 'read'] = True
    return refresh_alert_routing(updated)


def acknowledge_alerts(df, alert_ids):
    updated = df.copy()
    updated.loc[updated['alert_id'].isin(alert_ids), 'acknowledged'] = True
    updated.loc[updated['alert_id'].isin(alert_ids), 'read'] = True
    return refresh_alert_routing(updated)


def dismiss_alerts(df, alert_ids):
    updated = df.copy()
    updated.loc[updated['alert_id'].isin(alert_ids), 'dismissed'] = True
    return refresh_alert_routing(updated)


def insert_live_alert(df, template=None):
    next_id = len(df) + 1
    tpl = template or random.choice(ALERT_TEMPLATES)
    new_alert = {
        'event_id': f"LIVE-{next_id:04d}",
        'timestamp': now,
        'source': 'live_feed',
        'cat': tpl['cat'],
        'detected_sev': tpl['sev'],
        'title': tpl['title'],
        'body': tpl['body'],
        'location': 'Live Operations Desk',
        'confidence': 0.96,
        'impact_score': SEVERITY_CONFIG[tpl['sev']]['weight'],
        'affected_flights': 3 if tpl['sev'] in ['critical', 'high'] else 1,
    }
    row = pd.DataFrame([new_alert])
    row = pd.concat([row, row.apply(normalize_severity, axis=1)], axis=1)
    row['dedupe_key'] = row.apply(dedupe_key, axis=1)
    row['duplicate_count'] = 1
    row['duplicate_event_ids'] = row['event_id'].map(lambda event_id: [event_id])
    row['alert_id'] = f"ALT-{next_id:04d}"
    row['read'] = False
    row['acknowledged'] = False
    row['dismissed'] = False
    row['fresh'] = True
    row['age_min'] = 0.0
    combined = pd.concat([row, df], ignore_index=True)
    combined = combined.sort_values(['priority_score', 'timestamp'], ascending=[False, False]).reset_index(drop=True)
    return refresh_alert_routing(combined)


simulated_df = insert_live_alert(feed_df, template=ALERT_TEMPLATES[0])
critical_to_ack = simulated_df[simulated_df['requires_ack']].head(1)['alert_id'].tolist()
simulated_df = acknowledge_alerts(simulated_df, critical_to_ack)
simulated_df = mark_read(simulated_df, simulated_df.head(2)['alert_id'].tolist())
simulated_df = dismiss_alerts(simulated_df, simulated_df.tail(1)['alert_id'].tolist())
simulated_df[['alert_id', 'sev', 'cat', 'title', 'read', 'acknowledged', 'dismissed', 'fresh']].head(5)


## 9. Notification Analytics

In [ ]:
unread_df = feed_df[(~feed_df['read']) & (~feed_df['dismissed'])]
severity_counts = unread_df['sev'].value_counts().reindex(['critical', 'high', 'medium', 'low'], fill_value=0)
category_counts = unread_df['cat'].value_counts().reindex(CATEGORY_CONFIG.keys(), fill_value=0)
sla_breach_count = int((feed_df['escalation_status'] == 'escalate').sum())
dedupe_saved = int(scored_df.shape[0] - feed_df.shape[0])
avg_priority = round(feed_df['priority_score'].mean(), 1)
critical_unacknowledged = int(((feed_df['sev'] == 'critical') & (~feed_df['acknowledged']) & (~feed_df['dismissed'])).sum())

notification_kpis = {
    'total_events_ingested': int(scored_df.shape[0]),
    'active_alerts': int((~feed_df['dismissed']).sum()),
    'unread_alerts': int(unread_df.shape[0]),
    'critical_unread': int(severity_counts['critical']),
    'critical_unacknowledged': critical_unacknowledged,
    'high_unread': int(severity_counts['high']),
    'sla_breaches': sla_breach_count,
    'duplicates_suppressed': dedupe_saved,
    'average_priority_score': avg_priority,
}

notification_kpis


## 10. AI Notification Recommendations

In [ ]:
def build_notification_recommendations():
    recs = []
    if notification_kpis['critical_unacknowledged'] > 0:
        recs.append({
            'priority': 'critical',
            'area': 'acknowledgement',
            'action': 'Force acknowledgement workflow for critical alerts',
            'reason': f"{notification_kpis['critical_unacknowledged']} critical alerts still require acknowledgement.",
        })
    if notification_kpis['sla_breaches'] > 0 and not escalations_df.empty:
        top_breach = escalations_df.sort_values('priority_score', ascending=False).iloc[0]
        recs.append({
            'priority': 'high',
            'area': 'escalation',
            'action': f"Escalate {top_breach['alert_id']} to {top_breach['route_to']}",
            'reason': f"Alert age {top_breach['age_min']} min exceeds SLA {top_breach['sla_min']} min.",
        })
    hot_category = category_counts.sort_values(ascending=False).index[0]
    recs.append({
        'priority': 'medium',
        'area': 'routing',
        'action': f"Increase monitoring for {CATEGORY_CONFIG[hot_category]['label']}",
        'reason': f"{int(category_counts[hot_category])} unread alerts are concentrated in this category.",
    })
    noisiest = feed_df.sort_values('duplicate_count', ascending=False).head(1)
    if not noisiest.empty and int(noisiest.iloc[0]['duplicate_count']) > 1:
        recs.append({
            'priority': 'low',
            'area': 'dedupe',
            'action': 'Keep duplicate suppression enabled at 20 minutes',
            'reason': f"{int(noisiest.iloc[0]['duplicate_count'])} near-identical events were collapsed for {noisiest.iloc[0]['title']}.",
        })
    return pd.DataFrame(recs)

recommendations_df = build_notification_recommendations()
recommendations_df


## 11. Notification Visuals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

severity_counts.plot(kind='bar', ax=axes[0, 0], color=['#ef4444', '#f97316', '#fbbf24', '#34d399'])
axes[0, 0].set_title('Unread Alerts by Severity')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=0)

category_counts.plot(kind='bar', ax=axes[0, 1], color='#60a5fa')
axes[0, 1].set_title('Unread Alerts by Category')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=35)

timeline = feed_df.set_index('timestamp').sort_index().resample('15min')['alert_id'].count()
timeline.plot(ax=axes[1, 0], marker='o', color='#a78bfa')
axes[1, 0].set_title('Alert Volume Timeline')
axes[1, 0].set_ylabel('Alerts')

sla_counts = feed_df['escalation_status'].value_counts().reindex(['within_sla', 'escalate'], fill_value=0)
sla_counts.plot(kind='bar', ax=axes[1, 1], color=['#34d399', '#ef4444'])
axes[1, 1].set_title('SLA Status')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 12. Backend Payload

In [ ]:
def records_with_string_dates(df):
    copy = df.copy()
    for col in copy.select_dtypes(include=['datetime64[ns]']).columns:
        copy[col] = copy[col].astype(str)
    return copy.to_dict(orient='records')

payload = {
    'generated_at': now.isoformat(),
    'kpis': notification_kpis,
    'severity_counts': severity_counts.to_dict(),
    'category_counts': category_counts.to_dict(),
    'alerts': records_with_string_dates(feed_df.head(80)),
    'visible_alerts': records_with_string_dates(visible_alerts_df.head(40)),
    'escalations': records_with_string_dates(escalations_df),
    'recommendations': recommendations_df.to_dict(orient='records'),
    'ui_state': {
        'live': True,
        'enabled_severities': ['critical', 'high', 'medium', 'low'],
        'category_filter': 'all',
        'search': '',
    },
}

payload.keys()

## 13. Summary

In [ ]:
print('NOTIFICATION SUMMARY')
print('====================')
print(f"Events ingested: {notification_kpis['total_events_ingested']}")
print(f"Active alerts: {notification_kpis['active_alerts']}")
print(f"Unread alerts: {notification_kpis['unread_alerts']}")
print(f"Critical unread: {notification_kpis['critical_unread']}")
print(f"High unread: {notification_kpis['high_unread']}")
print(f"SLA breaches: {notification_kpis['sla_breaches']}")
print(f"Duplicates suppressed: {notification_kpis['duplicates_suppressed']}")

print('\nTop notification recommendations:')
for _, rec in recommendations_df.iterrows():
    print(f"- [{rec['priority']}] {rec['action']} - {rec['reason']}")